# Level 200: Geographic Routing with Azure API Management (Premium)

This notebook demonstrates geographic-based routing using Azure API Management (Premium SKU) with Traffic Manager. The setup includes:
- Primary region: East US
- Secondary region: Brazil South
- Premium tier APIM for multi-region support
- Traffic Manager with geographic routing

## Prerequisites
- Azure subscription with Premium tier API Management
- Traffic Manager profile configured for geographic routing
- Backend APIs deployed in East US and Brazil South regions

In [ ]:
import os
import json
import requests
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import networkx as nx
import pandas as pd

# Configuration
RESOURCE_GROUP = "apim-geo-demo-rg"
APIM_NAME = "apim-geo-demo"
TM_PROFILE = "apim-geo-demo-tm"

## Test Geographic Routing

Test how Traffic Manager routes requests based on client location using real Azure IP ranges. The Premium SKU APIM allows us to have multiple regional gateways, which Traffic Manager will route to based on the client's geographic location.

In [ ]:
def visualize_routing(client_location, target_region):
    """Create a network graph showing request routing path"""
    G = nx.DiGraph()
    
    # Node positions for visual layout
    positions = {
        'Client': (0, 0.5),
        'Traffic Manager': (0.5, 0.5),
        'East US': (1, 0.8),
        'Brazil South': (1, 0.2)
    }
    
    # Add nodes
    for node, pos in positions.items():
        G.add_node(node, pos=pos)
    
    # Add edges
    G.add_edge('Client', 'Traffic Manager')
    G.add_edge('Traffic Manager', target_region)
    
    # Create plot
    plt.figure(figsize=(12, 6))
    nx.draw(G, positions, with_labels=True, node_color='lightblue', 
            node_size=3000, arrowsize=20, font_size=10,
            font_weight='bold', arrows=True)
    
    plt.title(f'Request Route: {client_location} → {target_region}')
    plt.axis('off')
    plt.show()

def test_geographic_routing():
    """Interactive widget to test geographic routing"""
    location_dropdown = widgets.Dropdown(
        options=['North America', 'South America', 'Europe', 'Asia'],
        description='Client Location:',
        style={'description_width': 'initial'}
    )
    
    test_button = widgets.Button(description='Test Route')
    output = widgets.Output()
    
    def on_test_click(b):
        with output:
            output.clear_output()
            location = location_dropdown.value
            
            # Real Azure IP ranges for testing
            test_ips = {
                'North America': '23.96.0.0',    # Azure East US
                'South America': '191.233.0.0',  # Azure Brazil South
                'Europe': '13.80.0.0',          # Azure West Europe
                'Asia': '13.75.0.0'            # Azure Southeast Asia
            }
            test_ip = test_ips[location]
            
            # Expected routing based on geographic mapping
            expected_region = {
                'North America': 'East US',
                'South America': 'Brazil South',
                'Europe': 'East US',        # Default to primary
                'Asia': 'East US'          # Default to primary
            }
            target_region = expected_region[location]
            
            try:
                response = requests.get(
                    f'https://{TM_PROFILE}.trafficmanager.net',
                    headers={'X-Client-IP': test_ip},
                    timeout=10
                )
                data = response.json()
                
                visualize_routing(location, target_region)
                print(f'Request from: {location} (IP: {test_ip})')
                print(f'Routed to: {target_region}')
                print(f'Response: {data["message"]}')
                
            except Exception as e:
                print(f"Error making request: {str(e)}")
                print("Note: This is expected if the APIM service is not deployed")
    
    test_button.on_click(on_test_click)
    display(location_dropdown, test_button, output)

test_geographic_routing()

## Traffic Manager Configuration

Display the current Traffic Manager profile configuration showing how geographic routing is set up between East US and Brazil South regions.

In [ ]:
def display_tm_config():
    """Display Traffic Manager endpoint configuration"""
    endpoints = [
        {
            'Name': 'primary',
            'Region': 'East US',
            'Geographic Mapping': 'North America, Europe, Asia',
            'Type': 'Premium APIM Gateway',
            'Status': 'Enabled'
        },
        {
            'Name': 'secondary',
            'Region': 'Brazil South',
            'Geographic Mapping': 'South America',
            'Type': 'Premium APIM Gateway',
            'Status': 'Enabled'
        }
    ]
    
    df = pd.DataFrame(endpoints)
    display(HTML(df.to_html(index=False)))

display_tm_config()